# LLM-Based Parallel Data Construction (Proof of Concept)

This notebook tests the ability of a local LLM to translate original Shakespearean text into modern English while preserving semantic content. We use the paired 16-play dataset to establish a ground truth for evaluation.

**Goals:**
1. Generate modern English translations zero-shot using an Instruct LLM.
2. Evaluate content preservation using Semantic Similarity (Sentence-BERT), BLEU, and ROUGE metrics.

In [1]:
## Data Loading and Sampling
## We sample a subset to keep inference time manageable for the PoC.
import os
os.environ["TRITON_DISABLE_LINE_INFO"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
import random
from pathlib import Path

# Relative path from marthabot/code/notebooks to the data
DATA_DIR = Path("../../../shakespeare_data/align/plays/model_16plays")

modern_file = DATA_DIR / "train_plays.modern"
original_file = DATA_DIR / "train_plays.original"

modern_lines = []
original_lines = []

with open(modern_file, "r", encoding="utf-8") as fm, open(original_file, "r", encoding="utf-8") as fo:
    for mod, orig in zip(fm, fo):
        mod, orig = mod.strip(), orig.strip()
        if mod and orig:
            modern_lines.append(mod)
            original_lines.append(orig)

assert len(modern_lines) == len(original_lines), "Mismatched parallel data"
print(f"Total pairs loaded: {len(modern_lines)}")

# Sample 100 pairs for the proof of concept
random.seed(42)
SAMPLE_SIZE = 100
indices = random.sample(range(len(modern_lines)), SAMPLE_SIZE)

sample_modern = [modern_lines[i] for i in indices]
sample_original = [original_lines[i] for i in indices]

print(f"Sampled {SAMPLE_SIZE} pairs for evaluation.")

Total pairs loaded: 19617
Sampled 100 pairs for evaluation.


In [2]:
## Initialize the LLM
## Ensure you have requested a GPU node on the SCC.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Using Qwen 14B Instruct (adjust repo name if using a different variant)
MODEL_ID = "Qwen/Qwen3-14B"

print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto", # Automatically places weights on available GPUs
    attn_implementation="eager" 
)

print("Model loaded successfully.")

/projectnb/scottml/gamato23/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Qwen/Qwen3-14B...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 443/443 [00:03<00:00, 143.11it/s]


Model loaded successfully.


In [3]:
import re
from tqdm import tqdm

system_prompt = (
    "You are an expert translator. Translate the following Shakespearean text into plain, "
    "everyday modern English. Preserve the factual content and semantic meaning exactly, "
    "but completely remove the Shakespearean style, vocabulary, and grammar. "
    "Return ONLY the translated text, without any introduction, quotes, or explanation."
)

generated_modern = []

for text in tqdm(sample_original, desc="Translating"):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text}
    ]
    
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text_input], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1024, # Massive increase to guarantee it finishes thinking
            temperature=0.3,
            do_sample=True
        )
    
    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    raw_translation = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    
    # Strip out the <think>...</think> blocks using regex
    clean_translation = re.sub(r'<think>.*?</think>', '', raw_translation, flags=re.DOTALL).strip()
    
    # Fallback in case it still somehow forgets the closing tag
    if "<think>" in clean_translation:
        clean_translation = clean_translation.split("</think>")[-1].strip()
    generated_modern.append(clean_translation)

Translating: 100%|██████████| 100/100 [34:23<00:00, 20.63s/it]


In [4]:
## Evaluate Generation Quality
import numpy as np
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
import warnings
warnings.filterwarnings('ignore')

# 1. Semantic Similarity (Sentence-BERT)
print("Calculating Semantic Similarity...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings_ground_truth = sbert_model.encode(sample_modern, convert_to_tensor=True)
embeddings_generated = sbert_model.encode(generated_modern, convert_to_tensor=True)
cosine_scores = util.cos_sim(embeddings_generated, embeddings_ground_truth)
semantic_sims = [cosine_scores[i][i].item() for i in range(len(sample_modern))]

# 2. ROUGE & BLEU
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
rouge_scores = []
bleu_scores = []

for gt, gen in zip(sample_modern, generated_modern):
    # ROUGE-L
    r_score = scorer.score(gt, gen)['rougeL'].fmeasure
    rouge_scores.append(r_score)
    
    # BLEU
    gt_tokens = [gt.split()]
    gen_tokens = gen.split()
    b_score = sentence_bleu(gt_tokens, gen_tokens, weights=(0.5, 0.5, 0, 0)) # Bigram focus
    bleu_scores.append(b_score)

print("\n--- Average Metrics ---")
print(f"Semantic Similarity (SBERT): {np.mean(semantic_sims):.4f}")
print(f"ROUGE-L F1:                  {np.mean(rouge_scores):.4f}")
print(f"BLEU (Bigram):               {np.mean(bleu_scores):.4f}")

Calculating Semantic Similarity...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 16033.75it/s]



--- Average Metrics ---
Semantic Similarity (SBERT): 0.6540
ROUGE-L F1:                  0.4863
BLEU (Bigram):               0.2119


In [5]:
## Print Side-by-Side Comparisons

NUM_DISPLAY = 5
display_indices = random.sample(range(SAMPLE_SIZE), NUM_DISPLAY)

print("--- Qualitative Review ---\n")
for idx in display_indices:
    print(f"Original (Input)  : {sample_original[idx]}")
    print(f"Ground Truth      : {sample_modern[idx]}")
    print(f"LLM Generated     : {generated_modern[idx]}")
    print(f"SBERT Score       : {semantic_sims[idx]:.4f}")
    print("-" * 80)

--- Qualitative Review ---

Original (Input)  : I say thou hast belied mine innocent child.
Ground Truth      : You have ruined my innocent child.
LLM Generated     : I say you have slandered my innocent child.
SBERT Score       : 0.7242
--------------------------------------------------------------------------------
Original (Input)  : Be Kent unmannerly When Lear is mad.
Ground Truth      : Kent can speak rudely when Lear goes mad.
LLM Generated     : Let Kent be rude when Lear is mad.
SBERT Score       : 0.9375
--------------------------------------------------------------------------------
Original (Input)  : But hollow men, like horses hot at hand, Make gallant show and promise of their mettle.
Ground Truth      : But insincere men, like horses who are too lively at the start of a race, make a big show of their spirit.
LLM Generated     : But hollow men, like eager horses, make a bold show and promise to demonstrate their strength.
SBERT Score       : 0.6464
----------------------